# AmsiNet - Step 17: do flat classifiers also benefit from free context? (CPU, a few minutes)
**Question:** most of AmsiNet's gain comes from free word context, so what does a *flat* ViT-B/16 (and any other flat model) gain from the same context and cascade?
For every flat model that has saved probabilities, this notebook applies exactly the same pipeline as Step 9 (fusion weights and tau* fitted on validation, test once): the flat classifier alone, + free context, + cascade, + full fusion, with its own cost per word. The AmsiNet word agent (MoE and linear) is included for reference.
Flat models used: `flatvit` (ViT-B/16 from Step 2, already on Drive) and, if you have run Step 16, `flat_vits` (ViT-S/16) and `flat_effb0` (EfficientNet-B0, about the cost of the whole AmsiNet cascade).

In [ ]:
# 1. Setup
from google.colab import drive
import os
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: reconnect and run this cell again'
import json, numpy as np, pandas as pd
from scipy.special import softmax
from scipy import stats
ROOT = '/content/drive/MyDrive/amsinet'
CFG = dict(SEEDS=[0, 1, 2, 3, 4], VARIANTS=['word', 'word_moe'], EPS=1e-4, KEEP=0.90)
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8'))
SCRIPTS, INDEX = D['scripts'], D['index']; SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS); IDX = {r['id']: r for r in INDEX}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv'); REGDF = pd.read_csv(f'{ROOT}/data/regions_meta.csv')
eff = pd.read_csv(f'{ROOT}/agents/agent_efficiency_cpu.csv').set_index('agent')['GFLOPs']
flat_df = pd.read_csv(f'{ROOT}/results/backbone_efficiency.csv'); FLAT = float(flat_df[flat_df.agent == 'Flat baseline']['GFLOPs'].iloc[0])
missing = [f'{t}_seed{s}' for s in CFG['SEEDS'] for t in ('block', 'line', 'word', 'word_moe') if not os.path.exists(f'{ROOT}/agents/{t}_seed{s}_test_probs.npy')]
print('missing agent files:', missing or 'none')

In [ ]:
# 2. Helpers (same definitions as Steps 6 and 8)
def load(tag, sp):
    P = np.load(f'{ROOT}/agents/{tag}_{sp}_probs.npy').astype(np.float32); pa = pd.read_csv(f'{ROOT}/agents/{tag}_{sp}_paths.csv'); return P, pa.path.tolist()
def region_lookup(level, tag, sp):
    P, paths = load(tag, sp); pos = {p: i for i, p in enumerate(paths)}
    d = REGDF[(REGDF.level == level) & (REGDF.split == sp)]
    return {(r.image_id, r.rid): P[pos[r.path]] for r in d.itertuples() if r.path in pos}
def table(sp, wtag, ltag, btag):
    d = META[META.split == sp].reset_index(drop=True); n = len(d)
    Pw, pw = load(wtag, sp); assert pw == d.path.tolist(), 'word probabilities are not aligned with crops_meta.csv'
    logP = np.log(np.clip(Pw, CFG['EPS'], 1)); lid, bid = [], []
    for iid, k in zip(d.image_id, d.word_idx):
        w = IDX[iid]['words'][k]; lid.append(w.get('line', -1)); bid.append(w.get('block', -1))
    img = d.image_id.values; lkey = img.astype(str) + '|l' + np.array(lid).astype(str); bkey = img.astype(str) + '|b' + np.array(bid).astype(str)
    lg, lu = pd.factorize(lkey); bg, bu = pd.factorize(bkey)
    def loo(g, ng):
        S = np.zeros((ng, C)); cnt = np.zeros(ng); np.add.at(S, g, logP); np.add.at(cnt, g, 1); o = cnt[g] - 1
        F = np.zeros((n, C)); m = o > 0; F[m] = (S[g][m] - logP[m]) / o[m][:, None]; return F
    def reg(level, tag, key):
        L = region_lookup(level, tag, sp); F = np.zeros((n, C))
        for i, (iid, kk) in enumerate(zip(img, key)):
            v = L.get((iid, f'{level[0]}{kk}'))
            if v is not None: F[i] = np.log(np.clip(v, CFG['EPS'], 1))
        return F
    ui, inv = np.unique(img, return_inverse=True); l_img = np.zeros(len(lu), int); l_img[lg] = inv; b_img = np.zeros(len(bu), int); b_img[bg] = inv
    return dict(y=np.array([SIDX[s] for s in d.script]), word=logP, line_ctx=loo(lg, len(lu)), block_ctx=loo(bg, len(bu)),
                line_agent=reg('line', ltag, lid), block_agent=reg('block', btag, bid),
                img=img, inv=inv, n_img=len(ui), lg=lg, bg=bg, nl=len(lu), nb=len(bu), l_img=l_img, b_img=b_img)
def confmat(y, p): return np.bincount(y * C + p, minlength=C * C).reshape(C, C)
def f1_from(cm):
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; den = 2 * tp + fp + fn; return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
def metr(y, p): cm = confmat(y, p); return dict(acc=100 * float(np.trace(cm) / cm.sum()), f1=100 * float(f1_from(cm).mean()))
def unc(s): p = softmax(s, axis=1); return -(p * np.log(np.clip(p, 1e-12, 1))).sum(1) / np.log(C)
KEYS = ['line_ctx', 'block_ctx', 'line_agent', 'block_agent']; GRID_W = [0, .1, .25, .5, 1, 1.5, 2, 3]
def cascade(T, tau, W):
    s = T['word'] + W['line_ctx'] * T['line_ctx'] + W['block_ctx'] * T['block_ctx']; u = unc(s)
    li = np.zeros(T['nl'], bool); np.logical_or.at(li, T['lg'], u > tau); s = s + W['line_agent'] * T['line_agent'] * li[T['lg']][:, None]
    u = unc(s); bi = np.zeros(T['nb'], bool); np.logical_or.at(bi, T['bg'], u > tau); s = s + W['block_agent'] * T['block_agent'] * bi[T['bg']][:, None]
    return s, li, bi
def cost(T, li, bi, GF):
    nw = np.bincount(T['inv'], minlength=T['n_img']).astype(float); nl = np.bincount(T['l_img'][li], minlength=T['n_img']).astype(float); nb = np.bincount(T['b_img'][bi], minlength=T['n_img']).astype(float)
    return float((nw * GF['word'] + nl * GF['line'] + nb * GF['block']).mean())
TAUS = [0, .02, .05, .1, .15, .2, .3, .4, .5, .6, .7, .8, .9, 1.01]

## 3. Costs per word and the list of flat models that are available

In [ ]:
try:
    import torch
except ImportError: torch = None
GFW = {'word': float(eff['word']), 'word_moe': float(eff['word_moe']), 'flatvit': FLAT}
TAG = {'word': 'word', 'word_moe': 'word_moe', 'flatvit': 'flatvit'}
CAND = {'flat_vits': ('vit_small_patch16_224.augreg_in21k', 9.2), 'flat_effb0': ('efficientnet_b0.ra_in1k', 0.8), 'flat_bhunia': ('bhunia', None)}   # fallback GFLOPs if the counter is unavailable
for name, (mn, fb) in CAND.items():
    if all(os.path.exists(f'{ROOT}/agents/{name}_seed{s}_{sp}_probs.npy') for s in CFG['SEEDS'] for sp in ('val', 'test')):
        if name == 'flat_bhunia':
            GFW[name] = float(open(f'{ROOT}/agents/flat_bhunia_gflops.txt').read()); TAG[name] = name; print(f'{name}: {GFW[name]:.2f} GFLOPs per word'); continue
        try:
            import timm
            from torch.utils.flop_counter import FlopCounterMode
            m = timm.create_model(mn, pretrained=False, num_classes=C).eval(); x = torch.zeros(1, 3, 224, 224)
            with torch.no_grad(), FlopCounterMode(display=False) as fc: m(x)
            gf = fc.get_total_flops() / 1e9
        except Exception as e: print('FLOP counter failed for', name, e, '-> using', fb); gf = fb
        GFW[name] = gf; TAG[name] = name; print(f'{name}: {gf:.2f} GFLOPs per word')
    else: print('not available yet:', name)
VARS = [v for v in ['word_moe', 'word', 'flatvit', 'flat_vits', 'flat_effb0', 'flat_bhunia'] if v in GFW]; print('variants:', VARS)

## 4. One run per seed and model

In [ ]:
def run(seed, variant, keep_preds=False):
    wtag, ltag, btag = f'{TAG[variant]}_seed{seed}', f'line_seed{seed}', f'block_seed{seed}'
    GF = dict(word=GFW[variant], line=float(eff['line']), block=float(eff['block']))
    TV, TT = table('val', wtag, ltag, btag), table('test', wtag, ltag, btag)
    def full(T, W): return T['word'] + sum(W[k] * T[k] for k in KEYS)
    vf1 = lambda W: metr(TV['y'], full(TV, W).argmax(1))['f1']
    W = {k: 0.0 for k in KEYS}
    for _ in range(2):
        for k in KEYS: W[k] = max(GRID_W, key=lambda g: vf1({**W, k: g}))
    cv = {t: (lambda r: (metr(TV['y'], r[0].argmax(1))['f1'], cost(TV, r[1], r[2], GF)))(cascade(TV, t, W)) for t in TAUS}
    f_full, f_ctx = cv[0][0], cv[1.01][0]; target = f_ctx + CFG['KEEP'] * (f_full - f_ctx)
    ok = [t for t in TAUS if cv[t][0] >= target]; tau = min(ok, key=lambda t: cv[t][1]) if ok else 0.0
    out = dict(seed=seed, variant=variant, tau_star=tau, **{f'w_{k}': W[k] for k in KEYS})
    y = TT['y']; res = {}
    for name, (s, li, bi) in dict(word=(TT['word'], np.zeros(TT['nl'], bool), np.zeros(TT['nb'], bool)), context=cascade(TT, 1.01, W), cascade=cascade(TT, tau, W), full=cascade(TT, 0, W)).items():
        m = metr(y, s.argmax(1)); out[f'{name}_acc'], out[f'{name}_f1'], out[f'{name}_gflops'] = m['acc'], m['f1'], cost(TT, li, bi, GF); res[name] = s.argmax(1)
        if name == 'cascade': out['lines_pct'], out['blocks_pct'] = 100 * li.mean(), 100 * bi.mean()
    # mixed-script lines: lines whose words have more than one true script
    first = np.full(TT['nl'], -1); mixed = np.zeros(TT['nl'], bool)
    for g, yy in zip(TT['lg'], y):
        if first[g] < 0: first[g] = yy
        elif first[g] != yy: mixed[g] = True
    mw = mixed[TT['lg']]; out['mixed_line_pct'] = 100 * mixed.mean(); out['mixed_word_pct'] = 100 * mw.mean()
    for name in ('word', 'context', 'cascade', 'full'):
        out[f'{name}_acc_mixed'] = 100 * float((res[name][mw] == y[mw]).mean()); out[f'{name}_acc_single'] = 100 * float((res[name][~mw] == y[~mw]).mean())
    return (out, dict(y=y, **res, mw=mw, conf=np.exp(TT['word']).max(1), nline=np.bincount(TT['lg'])[TT['lg']], paths=META[META.split == 'test'].path.tolist())) if keep_preds else (out, None)

In [ ]:
rows = []
for variant in VARS:
    for seed in CFG['SEEDS']:
        o, _ = run(seed, variant); rows.append(o)
        print(variant, seed, 'tau*', o['tau_star'], '| alone %.2f/%.2f -> +context %.2f/%.2f -> cascade %.2f/%.2f (%.1f GF) | full %.2f/%.2f (%.1f GF)' % (o['word_acc'], o['word_f1'], o['context_acc'], o['context_f1'], o['cascade_acc'], o['cascade_f1'], o['cascade_gflops'], o['full_acc'], o['full_f1'], o['full_gflops']))
PS = pd.DataFrame(rows); PS.to_csv(f'{ROOT}/results/flat_plus_context_per_seed.csv', index=False)

## 5. Summary (mean +/- 95% CI over seeds)  -  paste this table back

In [ ]:
def mci(x):
    x = np.asarray(x, float); n = len(x); return x.mean(), (stats.t.ppf(.975, n - 1) * x.std(ddof=1) / np.sqrt(n) if n > 1 else float('nan'))
fmt = lambda x, d=2: '%.*f +/- %.*f' % (d, *(lambda m, h: (d, m, d, h))(*mci(x))) if False else ('%.*f +/- %.*f' % (d, mci(x)[0], d, mci(x)[1]))
NAMES = {'word_moe': 'AmsiNet word agent (EfficientViT-B0, MoE)', 'word': 'AmsiNet word agent (EfficientViT-B0, linear)', 'flatvit': 'flat ViT-B/16', 'flat_vits': 'flat ViT-S/16', 'flat_effb0': 'flat EfficientNet-B0', 'flat_bhunia': 'flat CNN + attention BiLSTM (Bhunia-style)'}
T = []
for v in VARS:
    d = PS[PS.variant == v]
    for k, lab in (('word', 'alone'), ('context', '+ free context'), ('cascade', '+ cascade at tau*'), ('full', '+ line and block agents everywhere')):
        T.append((NAMES[v], lab, fmt(d[f'{k}_acc']), fmt(d[f'{k}_f1']), fmt(d[f'{k}_gflops'], 1)))
S = pd.DataFrame(T, columns=['classifier', 'system', 'accuracy %', 'macro-F1 %', 'GFLOPs / image']); display(S)
S.to_csv(f'{ROOT}/results/flat_plus_context_summary.csv', index=False)

## 6. Paired comparison with the AmsiNet cascade (per-seed differences, same seeds)

In [ ]:
ref = PS[PS.variant == 'word_moe'].set_index('seed')
for v in VARS:
    if v == 'word_moe': continue
    d = PS[PS.variant == v].set_index('seed')
    for k in ('context', 'cascade'):
        da = (d[f'{k}_acc'] - ref['cascade_acc']).values; df = (d[f'{k}_f1'] - ref['cascade_f1']).values
        print(f"{NAMES[v]} {k} minus AmsiNet cascade (MoE): accuracy {mci(da)[0]:+.2f} +/- {mci(da)[1]:.2f} | macro-F1 {mci(df)[0]:+.2f} +/- {mci(df)[1]:.2f} | GFLOPs {d[f'{k}_gflops'].mean():.1f} vs {ref['cascade_gflops'].mean():.1f}")